# SQL Clinical Analysis
## MIMIC-IV Clinical Database Demo

### Overview
This notebook extends the exploratory analysis from notebook 01 by querying the 
cleaned admissions and diagnosis data using SQL. Four clinical questions are answered 
using SQLite, demonstrating structured querying skills on real hospital data.

### Clinical Questions
1. What is the admission volume and average length of stay by admission type?
2. What is the mortality rate by admission type?
3. What are the top 10 primary diagnoses driving admissions?
4. What is the 30-day readmission rate?

In [2]:
# Import required libraries
import pandas as pd
import sqlite3

In [3]:
# Load cleaned data
merged = pd.read_csv('../data/cleaned/merged_admissions.csv')
conn = sqlite3.connect(':memory:')

# Load dataframe into SQL table
merged.to_sql('admissions', conn, if_exists='replace', index=False)

print("Database ready")

Database ready


### Query 1: Admission Volume and Length of Stay by Type
Understanding which admission types drive the most volume and consume the most 
bed days is fundamental to hospital resource planning.

In [4]:
# Query 1: Count of admissions by admission type
query1 = """
SELECT admission_type, 
       COUNT(*) as total_admissions,
       ROUND(AVG(length_of_stay), 2) as avg_length_of_stay
FROM admissions
GROUP BY admission_type
ORDER BY total_admissions DESC
"""

result1 = pd.read_sql_query(query1, conn)
result1

,admission_type,total_admissions,avg_length_of_stay
0,EW EMER.,104,7.28
1,OBSERVATION ADMIT,45,8.15
2,URGENT,38,9.85
3,EU OBSERVATION,30,0.92
4,SURGICAL SAME DAY ADMISSION,18,5.70
5,DIRECT EMER.,15,9.39
6,ELECTIVE,13,8.19
7,DIRECT OBSERVATION,7,1.45
8,AMBULATORY OBSERVATION,5,1.02


Emergency walk-in admissions (EW EMER.) dominate volume at 104 cases, but urgent admissions have the longest average stay at 9.85 days. This confirms that high 
volume and high resource consumption are not always the same admission type.

### Query 2: Mortality Rate by Admission Type
Mortality rate by admission type reveals which patient groups carry the highest 
clinical risk; critical for prioritizing intervention and staffing decisions.

In [5]:
# Query 2: Mortality rate by admission type
query2 = """
SELECT admission_type,
       COUNT(*) as total_admissions,
       SUM(hospital_expire_flag) as total_deaths,
       ROUND(AVG(hospital_expire_flag) * 100, 2) as mortality_rate_pct
FROM admissions
GROUP BY admission_type
ORDER BY mortality_rate_pct DESC
"""

result2 = pd.read_sql_query(query2, conn)
result2

,admission_type,total_admissions,total_deaths,mortality_rate_pct
0,URGENT,38,5,13.16
1,OBSERVATION ADMIT,45,3,6.67
2,DIRECT EMER.,15,1,6.67
3,EW EMER.,104,6,5.77
4,SURGICAL SAME DAY ADMISSION,18,0,0.00
5,EU OBSERVATION,30,0,0.00
6,ELECTIVE,13,0,0.00
7,DIRECT OBSERVATION,7,0,0.00
8,AMBULATORY OBSERVATION,5,0,0.00


URGENT admissions have the highest mortality rate at 13.16%, more than double 
the emergency walk-in rate of 5.77%. Elective, observation, and ambulatory 
admissions recorded zero deaths, reflecting their lower acuity. Hospitals should 
treat urgent admissions with the same clinical urgency as emergencies.

In [14]:
# Load diagnosis tables
diagnoses = pd.read_csv('../data/raw/mimic-iv-clinical-database-demo-2.2/hosp/diagnoses_icd.csv.gz')
d_icd = pd.read_csv('../data/raw/mimic-iv-clinical-database-demo-2.2/hosp/d_icd_diagnoses.csv.gz')

print(diagnoses.shape)
print(d_icd.shape)
diagnoses.head()

(4506, 5)
(109775, 3)


,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10035185,22580999,3,4139,9
1,10035185,22580999,10,V707,9
2,10035185,22580999,1,41401,9
3,10035185,22580999,9,3899,9
4,10035185,22580999,11,V8532,9


In [7]:
# Load diagnosis tables into SQL
diagnoses.to_sql('diagnoses', conn, if_exists='replace', index=False)
d_icd.to_sql('d_icd', conn, if_exists='replace', index=False)

print("Diagnosis tables loaded")

Diagnosis tables loaded


### Query 3: Top 10 Primary Diagnoses Driving Admissions
Identifying the diagnoses most responsible for admissions helps hospitals plan 
specialty staffing, bed allocation, and treatment pathway investments.

In [8]:
# Query 3: Top 10 primary diagnoses driving admissions
query3 = """
SELECT d.long_title as diagnosis,
       COUNT(*) as total_cases,
       ROUND(AVG(a.length_of_stay), 2) as avg_length_of_stay
FROM diagnoses dx
JOIN d_icd d ON dx.icd_code = d.icd_code AND dx.icd_version = d.icd_version
JOIN admissions a ON dx.hadm_id = a.hadm_id
WHERE dx.seq_num = 1
GROUP BY d.long_title
ORDER BY total_cases DESC
LIMIT 10
"""

result3 = pd.read_sql_query(query3, conn)
result3

,diagnosis,total_cases,avg_length_of_stay
0,Coronary atherosclerosis of native coronary ar...,7,4.90
1,"Acute kidney failure, unspecified",7,3.02
2,Other postoperative infection,4,5.32
3,Non-ST elevation (NSTEMI) myocardial infarction,4,4.62
4,"Cerebral aneurysm, nonruptured",4,5.30
5,Aortic valve disorders,4,4.37
6,"Sepsis, unspecified organism",3,10.12
7,Other encephalopathy,3,9.19
8,Encounter for antineoplastic chemotherapy,3,8.54
9,Atrial fibrillation,3,5.08


Coronary atherosclerosis and acute kidney failure are the most frequent primary 
diagnoses at 7 cases each. However, sepsis records the longest average stay at 
10.12 days despite only 3 cases, followed by encephalopathy at 9.19 days. Volume 
alone does not reflect resource burden — high acuity diagnoses like sepsis consume 
disproportionate clinical resources per admission.

In [10]:
# Query 4: 30-day readmission analysis (excluding same-day transfers < 24 hours)
query4 = """
SELECT a1.subject_id,
       a1.hadm_id as first_admission,
       a1.dischtime as discharge_time,
       a2.hadm_id as readmission,
       a2.admittime as readmission_time,
       ROUND(JULIANDAY(a2.admittime) - JULIANDAY(a1.dischtime), 2) as days_between
FROM admissions a1
JOIN admissions a2 ON a1.subject_id = a2.subject_id
WHERE a2.admittime > a1.dischtime
AND JULIANDAY(a2.admittime) - JULIANDAY(a1.dischtime) <= 30
AND JULIANDAY(a2.admittime) - JULIANDAY(a1.dischtime) > 1
ORDER BY days_between ASC
"""

result4 = pd.read_sql_query(query4, conn)
print(f"Total 30-day readmissions: {len(result4)}")
result4.head(10)

Total 30-day readmissions: 58


,subject_id,first_admission,discharge_time,readmission,readmission_time,days_between
0,10002428,28662225,2156-04-29 16:26:00,20321825,2156-04-30 20:35:00,1.17
1,10004235,25970245,2196-06-19 14:54:00,22187210,2196-06-20 21:11:00,1.26
2,10014354,23132022,2148-06-28 13:54:00,27487226,2148-06-30 01:09:00,1.47
3,10002930,28301173,2197-04-15 12:01:00,25282382,2197-04-17 02:01:00,1.58
4,10014354,26228185,2150-05-07 14:10:00,24357615,2150-05-09 16:09:00,2.08
5,10007795,28477357,2136-05-02 16:35:00,25135483,2136-05-04 20:20:00,2.16
6,10019003,26529390,2155-05-19 18:27:00,25508812,2155-05-22 21:46:00,3.14
7,10014354,29600294,2148-08-18 21:12:00,26486158,2148-08-22 15:18:00,3.75
8,10039708,24928679,2143-09-22 23:00:00,20093566,2143-09-26 18:24:00,3.81
9,10014354,27487226,2148-07-13 19:35:00,27562275,2148-07-18 02:31:00,4.29


### Finding: 30-Day Readmission Rate
After excluding 3 same-day cases likely representing transfers or data entry errors 
(readmitted within 24 hours of discharge), the adjusted 30-day readmission rate is 
21.1% (58 out of 275 admissions). This exceeds the US national benchmark of 15%, 
suggesting potential gaps in discharge planning or post-discharge follow-up care. 
Further investigation into which diagnoses and admission types drive readmissions 
is recommended.

## Next Step
Visualize key findings in a Power BI dashboard for operational decision making
 

In [16]:
# Export SQL results for Power BI
result1.to_csv('../data/cleaned/admissions_by_type.csv', index=False)
result2.to_csv('../data/cleaned/mortality_by_type.csv', index=False)
result3.to_csv('../data/cleaned/top_diagnoses.csv', index=False)
result4.to_csv('../data/cleaned/readmissions.csv', index=False)

print("All results exported successfully")

All results exported successfully
